# OCR do Livro de Registro de Leis nº 1 (1892–1894): resultados

Este caderno mostra o pipeline final aplicado às **12 páginas** transcritas (páginas 5 a 16). Ele não refaz nenhum experimento. Todas as escolhas abaixo já foram decididas antes e estão em `configs/`. Para cada página, o caderno mostra:

1. cada operação de imagem aplicada **sobre a original**, em sequência, e a segmentação de linhas desenhada sobre a imagem;
2. cada etapa de texto (reconhecedor → reescore CTC → convenções → hifenização → correção léxica) com o **WER contra o GT**;
3. o texto final lado a lado com o GT, com as palavras erradas marcadas, e o recorte de cada linha ao lado do texto lido dela.

O processo completo, com todos os experimentos, está em `PROCESSO.md`.

## Resultado em uma tabela

WER normalizado (minúsculas, sem pontuação), soma das edições do corpus. Quanto menor, melhor.

| configuração | o que é | páginas de ajuste (9) | páginas reservadas (7, 10, 12) |
|---|---|---:|---:|
| baseline antiga | nlbin na segmentação e no reconhecimento | 70,54 | – |
| imagem crua | sem tratamento | 63,15 | – |
| v2 | seg. nlbin + convenções + léxico + reescore CTC | 47,74 | 47,87 |
| **v4** (`configs/best_v4.json`) | **busca gulosa condicional, modelo base** | **43,17** | **43,94** |
| **v3** (`configs/best_v3_ft.json`) | **v2 com o reconhecedor ajustado (ft_v1)** | (treinou nelas) | **29,61** |

- **v4** é o melhor resultado **sem treinar pesos**. Como não usa o GT da página avaliada, o número vale para as 12 páginas.
- **v3** é o melhor resultado de todos. O ft_v1, porém, foi treinado com as 9 páginas de ajuste, então só o número nas reservadas é honesto. Nas páginas de ajuste ele aparece com `*` (dentro da amostra).
- A meta do projeto era WER normalizado ≤ 50%.

## O que foi testado e o que ficou

### Arquitetura

Três fluxos de imagem compartilham a mesma geometria:

```
original → geometria (ampliação, inclinação)  = base (define as coordenadas)
base → operações da segmentação   → segmentador neural blla (Kraken) → linhas (polígonos)
base → operações do reconhecimento → PP-OCRv6 medium (Kraken) → texto → pós-processamento
```

As operações fotométricas não movem pixels, então os polígonos achados numa imagem valem na outra. Isso permite **segmentar numa imagem e reconhecer em outra**. O primeiro ganho veio daí: binarizar ajudava a segmentação, mas o reconhecedor perdia 9,6 pontos lendo pixels binarizados.

### Progressão nas páginas de ajuste (WER norm. %)

| etapa | mudança | WER |
|---|---|---:|
| baseline antiga | nlbin na segmentação **e** no reconhecimento | 70,54 |
| crua | sem tratamento | 63,15 |
| fluxos separados | segmenta no nlbin, reconhece na original | 60,95 |
| + convenções e hifenização | `n^o→nº`, `⁋→§`, junta palavra partida por `-` | 59,20 |
| + correção léxica (v1) | léxico do domínio (leave-one-out) + pt-BR 50k, distância de edição | 49,04 |
| + reescore CTC (v2) | escolhe o candidato do léxico pelas probabilidades do reconhecedor | 47,74 |
| busca gulosa condicional (v4) | todas as etapas redecididas em conjunto | **43,17** |

### Como a v4 foi encontrada: busca gulosa condicional

Testar cada mudança isolada sobre a crua e somar as que ganham não funciona, porque os efeitos interagem. A busca segue estas regras:

- As etapas são decididas em ordem, sempre **sobre o melhor conjunto atual**.
- Em cada etapa, cada candidato é testado sozinho. O melhor entra só se superar a referência.
- Depois, **todos os outros** candidatos da etapa são testados por cima do vencedor, inclusive os que pioraram sozinhos. Cada um fica só se melhorar.
- Um passe para trás remove qualquer escolha cuja retirada não piore.
- Os passes se repetem até nada mudar. Nenhuma decisão é definitiva.

Foram 3 passes nas 9 páginas de ajuste, 330 configurações avaliadas.

| etapa | passe 1: escolha | WER antes → depois | passe 2: escolha | WER antes → depois |
|---|---|---|---|---|
| reconhecedor | medium (small piora) | 63,15 → 63,15 | medium | 44,24 → 44,24 |
| imagem de segmentação | nlbin + contraste | 63,15 → 60,67 | **iluminação + CLAHE** | 44,24 → 43,62 |
| geometria | ampliação 1,5 | 60,67 → 59,65 | ampliação 1,5 + inclinação | 43,62 → 43,57 |
| filtro de linhas | linhas < 5% da largura | 59,65 → 59,59 | nenhum | 43,57 → 43,57 |
| imagem de reconhecimento | iluminação + nitidez + gama 1,5 | 59,59 → 57,51 | iluminação + gama 0,7 | 43,57 → 43,34 |
| texto | convenções + hifenização | 57,51 → 56,38 | igual | – |
| léxico | domínio + pt-BR | 56,38 → 50,34 | igual | – |
| parâmetros do léxico | razão 0,5, tamanho mínimo 3, 50k inteiro | 50,34 → 48,48 | igual | – |
| decodificador | ctc+edit | 48,48 → 47,35 | igual | – |
| parâmetros do decodificador | α = 16 | 47,35 → 46,78 | **α = 2** | 43,34 → 43,17 |

O que a busca mostrou:

- **O contraste piorou sozinho (+0,73) e ajudou sobre o nlbin.** Somando só o que melhora isoladamente, essa combinação nunca seria testada.
- **No passe 2 a segmentação trocou nlbin por iluminação + CLAHE**, depois que o reconhecimento e o léxico já estavam ativos. O nlbin, base de todas as versões anteriores, saiu.
- Tratamentos da imagem de reconhecimento, que isoladamente ficavam no ruído, passaram a ajudar quando decididos em conjunto.

**Quanto cada escolha da v4 vale** (WER ao retirar só ela; v4 = 43,17):

| retirar | WER | custo |
|---|---:|---:|
| léxico pt-BR | 51,52 | +8,35 |
| iluminação no reconhecimento | 48,25 | +5,08 |
| léxico do domínio | 46,95 | +3,78 |
| ampliação 1,5 | 45,20 | +2,03 |
| reescore ctc+edit | 45,20 | +2,03 |
| hifenização | 44,58 | +1,41 |
| léxico inteiro (50k) | 44,36 | +1,19 |
| CLAHE na segmentação | 43,91 | +0,73 |
| razão 0,5 | 43,85 | +0,68 |
| convenções | 43,74 | +0,56 |
| inclinação | 43,68 | +0,51 |
| gama 0,7 | 43,68 | +0,51 |
| iluminação na segmentação | 43,28 | +0,11 |
| tamanho mínimo 3 | 43,28 | +0,11 |
| α = 2 | 43,23 | +0,06 |

Nas bordas da grade, um parâmetro por vez, nada melhorou mais que ruído. Testados: α 1 e 4, razão 0,45 e 0,6, gama 0,5 e 0,85, ampliação 1,25 e 1,75, e 20 candidatos CTC. O melhor, gama 0,5, ganhou 0,34 ponto, cerca de 6 palavras em 1.772.

### Reconhecedores

| reconhecedor | resultado |
|---|---|
| PP-OCRv6 small | pior que o medium em todas as páginas (≈ +4 pontos) |
| **ft_v1**: medium ajustado com as linhas das 9 páginas de ajuste (50 épocas) | reservadas: 47,87 → **29,61**; CER cai de 19,2 para 10,0 |
| ft_v2: o mesmo com 100 épocas | 30,08 nas reservadas: sobreajuste |
| TraPrInq (Inquisição portuguesa, séc. XVI–XIX), sem ajuste | CER 38,8 contra 22,0 do medium |
| McCATMuS (multilíngue, séc. XVI–XXI), sem ajuste | CER 51,2 |
| TraPrInq ajustado com os mesmos dados do ft_v1 | 30,39 nas reservadas: empata com o ft_v1, sem superar |

- Não existe modelo público para manuscrito brasileiro do século XIX.
- O ft_v1 não aceita as imagens tratadas da v4. Ele foi treinado com recortes da imagem crua, e com iluminação + gama vai a 38,90. Por isso a v3 mantém o reconhecimento na imagem crua.
- Um conjunto por linha entre ft_v1 e TraPrInq ajustado deu 28,50 nas reservadas. É um ganho de ~7 palavras, medido uma única vez, e **não** entrou no pipeline.

### Limites

- Só 3 páginas reservadas, ~640 palavras: 1 ponto de WER ≈ 6 palavras.
- O GT não foi verificado e tem erros prováveis (p. ex. `com causa` contra `sem causa`). Parte do erro restante é da referência.
- A página 5 é outro recorte (521×543 px, 45 palavras), e a segmentação dela falha.

## Configurações finais

| etapa | v4 (`configs/best_v4.json`) | v3 (`configs/best_v3_ft.json`) |
|---|---|---|
| geometria | ampliação ×1,5 → correção de inclinação (±5°, passo 0,25°) | nenhuma |
| imagem de segmentação | normalização de iluminação (σ=25) → CLAHE (clip 2,0, grade 8) | binarização nlbin (limiar 0,5) |
| segmentador | blla (padrão do Kraken), sem filtro de linhas | blla |
| imagem de reconhecimento | normalização de iluminação (σ=25) → gama 0,7 | original |
| reconhecedor | PP-OCRv6 medium, padding 16 | ft_v1, padding 16 |
| reescore CTC | α = 2, 10 candidatos | α = 16, 10 candidatos |
| convenções e hifenização | sim | sim |
| léxico | domínio (GT das páginas de ajuste, sem a página avaliada) + pt-BR 50k inteiro | igual |
| correção por edição | razão 0,5, palavras com ≥ 3 letras | igual |

O léxico do domínio nunca contém o GT da página avaliada (*leave-one-out*). Nas reservadas usa só o GT das 9 páginas de ajuste.

## Preparação

In [ ]:
# Ambiente. No Colab: monta o Drive, entra na pasta e instala as dependências (o torch do Colab é mantido).
import os
import subprocess
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/entrega_ocr')  # ajuste se a pasta estiver em outro lugar
    subprocess.run([sys.executable, 'setup_pipeline.py'], check=True)
print('pasta de trabalho:', Path.cwd())

In [ ]:
import logging
import warnings

import torch
from IPython.display import HTML, display

from ocr_pipeline.config import load_config
from ocr_pipeline.data import discover, split
from ocr_pipeline.engine import Engine
from ocr_pipeline.presentation import page_html, stage_corpus, style, summary_html
from ocr_pipeline.visualize import bar_svg

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings('ignore', module='coremltools')  # aviso inofensivo do Kraken no macOS
PAGINAS = discover(Path('imagens'), Path('ground_truth'))
AJUSTE, RESERVADAS = split(PAGINAS)  # semente 42: reservadas = 7, 10, 12
RESERVADAS_N = {p.number for p in RESERVADAS}
CONFIGS = {'v4 (medium, sem treino)': load_config(Path('configs/best_v4.json'))}
if Path('models/finetuned/ft_v1.safetensors').is_file():
    CONFIGS['v3 (ft_v1 ajustado)'] = load_config(Path('configs/best_v3_ft.json'))
else:
    print('ft_v1 ausente em models/finetuned/: mostrando só a v4')
AJUSTADOS = {nome for nome, config in CONFIGS.items() if config.rec.model == 'ft_v1'}
DISPOSITIVO = 'cuda:0' if torch.cuda.is_available() else 'cpu'
ENGINE = Engine(Path('cache'), DISPOSITIVO, seed=42)
RESULTADOS = {nome: {} for nome in CONFIGS}


def mostrar_html(corpo: str) -> None:
    display(HTML(f'{style()}<div class="ocr-body">{corpo}</div>'))


def mostrar(numero: int) -> None:
    """Roda as configurações na página e mostra imagens, etapas de texto e WER."""
    pagina = next(p for p in PAGINAS if p.number == numero)
    partes = []
    for nome, config in CONFIGS.items():
        aviso = ''
        if nome in AJUSTADOS and numero not in RESERVADAS_N:
            aviso = 'Atenção: o ft_v1 foi treinado com linhas desta página; o resultado é otimista (dentro da amostra).'
        corpo, etapas = page_html(ENGINE, pagina, config, AJUSTE, f'{nome}: página {numero}', aviso)
        RESULTADOS[nome][numero] = etapas
        partes.append(corpo)
    mostrar_html(''.join(partes))


print('páginas:', [p.number for p in PAGINAS])
print('ajuste:', [p.number for p in AJUSTE], '| reservadas:', sorted(RESERVADAS_N))
print('dispositivo:', DISPOSITIVO, '| configurações:', list(CONFIGS))

## Página por página

Cada célula roda as configurações em uma página. A primeira execução leva de 20 a 60 s por página em CPU; os resultados ficam em `cache/` e as execuções seguintes são imediatas.

Como ler cada cartão:

- **geometria / imagem de segmentação / imagem de reconhecimento**: cada imagem é a anterior com mais uma operação, e a legenda dá o valor do parâmetro. A última imagem da segmentação mostra as linhas encontradas (polígono colorido + linha de base, numeradas na ordem de leitura).
- **texto: cada etapa**: WER e CER depois de cada operação de texto, e Δ em relação à etapa anterior.
- **texto final contra o GT**: <span style="text-decoration:line-through">riscado</span> = palavra errada, em verde = o que o GT diz. Ao lado está o GT.
- **linha a linha** (expandir): recorte da imagem de reconhecimento ↔ texto lido ↔ mudanças do reescore CTC.

### Página 5 (ajuste)

In [ ]:
mostrar(5)

### Página 6 (ajuste)

In [ ]:
mostrar(6)

### Página 7 (reservada)

In [ ]:
mostrar(7)

### Página 8 (ajuste)

In [ ]:
mostrar(8)

### Página 9 (ajuste)

In [ ]:
mostrar(9)

### Página 10 (reservada)

In [ ]:
mostrar(10)

### Página 11 (ajuste)

In [ ]:
mostrar(11)

### Página 12 (reservada)

In [ ]:
mostrar(12)

### Página 13 (ajuste)

In [ ]:
mostrar(13)

### Página 14 (ajuste)

In [ ]:
mostrar(14)

### Página 15 (ajuste)

In [ ]:
mostrar(15)

### Página 16 (ajuste)

In [ ]:
mostrar(16)

## Resumo das 12 páginas

WER normalizado final por página. `*` = página usada no treino do ft_v1 (resultado dentro da amostra). O gráfico mostra o WER do corpus depois de cada etapa de texto: v4 em todas as páginas, v3 só nas reservadas.

In [ ]:
mostrar_html('<h3>WER norm. final (%)</h3>' + summary_html(RESULTADOS, RESERVADAS_N, AJUSTADOS))
for nome in CONFIGS:
    paginas = sorted(RESERVADAS_N) if nome in AJUSTADOS else sorted(RESULTADOS[nome])
    etapas = stage_corpus(RESULTADOS[nome], paginas)
    mostrar_html(f'<h3>{nome}: WER norm. do corpus por etapa (páginas {paginas})</h3>'
                 + bar_svg({rotulo: 100 * s['wer_norm'] for rotulo, s in etapas}))

# textos finais em resultados/textos/<configuração>/pagina_NN.txt
for nome, paginas in RESULTADOS.items():
    pasta = Path('resultados/textos') / nome.split()[0]
    pasta.mkdir(parents=True, exist_ok=True)
    for numero, etapas in paginas.items():
        (pasta / f'pagina_{numero:02d}.txt').write_text(etapas[-1].text, encoding='utf-8')
print('textos salvos em resultados/textos/')